In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

FREIHAND_RGB_DIR = Path("data/raw/freihand/training/rgb")
LANDMARKS_PATH = Path("data/processed/freihand/landmarks_2d.npy")
SPLITS_PATH = Path("data/splits/freihand_splits.json")
IMAGE_SIZE = 224
INDEX_FINGERTIP = 8

# Landmark connectivity for drawing the hand skeleton
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # index
    (0, 9), (9, 10), (10, 11), (11, 12),   # middle
    (0, 13), (13, 14), (14, 15), (15, 16), # ring
    (0, 17), (17, 18), (18, 19), (19, 20), # pinky
    (5, 9), (9, 13), (13, 17),             # palm
]

In [ ]:
landmarks = np.load(LANDMARKS_PATH)
with SPLITS_PATH.open("r", encoding="utf-8") as f:
    splits = json.load(f)

print(f"Landmarks array shape: {landmarks.shape}")
print(f"Train: {len(splits['train']):,} images")
print(f"Val:   {len(splits['val']):,} images")
print(f"Test:  {len(splits['test']):,} images")

# Overlap check
train_set = set(splits["train"])
val_set = set(splits["val"])
test_set = set(splits["test"])
assert not (train_set & val_set), "Train/val overlap detected"
assert not (train_set & test_set), "Train/test overlap detected"
assert not (val_set & test_set), "Val/test overlap detected"
print("\n[OK] No split overlap detected.")

In [ ]:
def draw_hand(ax, image: np.ndarray, uv: np.ndarray, title: str) -> None:
    """Plot an image with overlaid hand landmarks and skeleton."""
    ax.imshow(image)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

    # Scale normalized coordinates back to pixel space.
    uv_px = uv * IMAGE_SIZE

    # Draw skeleton connections.
    for a, b in HAND_CONNECTIONS:
        ax.plot(
            [uv_px[a, 0], uv_px[b, 0]],
            [uv_px[a, 1], uv_px[b, 1]],
            color="#4A90D9",
            linewidth=0.8,
            alpha=0.7,
        )

    # Draw all landmarks as small dots.
    ax.scatter(
        uv_px[:, 0],
        uv_px[:, 1],
        s=12,
        c="#FFFFFF",
        edgecolors="#222222",
        linewidths=0.4,
        zorder=5,
    )

    # Highlight index fingertip (landmark 8) in red.
    ax.scatter(
        uv_px[INDEX_FINGERTIP, 0],
        uv_px[INDEX_FINGERTIP, 1],
        s=40,
        c="#E84040",
        edgecolors="#FFFFFF",
        linewidths=0.8,
        zorder=6,
    )

In [ ]:
NUM_SAMPLES = 5
rng = np.random.default_rng(42)

fig, axes = plt.subplots(
    nrows=3,
    ncols=NUM_SAMPLES,
    figsize=(NUM_SAMPLES * 3, 3 * 3),
)
fig.suptitle(
    "FreiHAND samples - white dots: all landmarks, red dot: index fingertip (landmark 8)",
    fontsize=10,
    y=1.01,
)

for row_idx, split_name in enumerate(["train", "val", "test"]):
    split_indices = splits[split_name]
    chosen = rng.choice(split_indices, size=NUM_SAMPLES, replace=False)

    for col_idx, img_idx in enumerate(chosen):
        img_path = FREIHAND_RGB_DIR / f"{img_idx:08d}.jpg"
        image = np.array(Image.open(img_path).convert("RGB"))
        uv = landmarks[img_idx]

        ax = axes[row_idx, col_idx]
        draw_hand(ax, image, uv, title=f"{split_name} #{img_idx}")

        if col_idx == 0:
            ax.set_ylabel(split_name.upper(), fontsize=11, fontweight="bold", labelpad=8)

plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/freihand_sanity_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Figure saved to report/figures/freihand_sanity_check.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Index fingertip (landmark 8) coordinate distributions per split", fontsize=11)

for ax, split_name in zip(axes, ["train", "val", "test"]):
    idx = splits[split_name]
    tips = landmarks[idx, INDEX_FINGERTIP, :]

    ax.scatter(tips[::50, 0], tips[::50, 1], alpha=0.15, s=2, c="#4A90D9")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_xlabel("x (normalized)")
    ax.set_ylabel("y (normalized)")
    ax.set_title(f"{split_name} (n={len(idx):,})")
    ax.set_aspect("equal")

    print(
        f"{split_name:5s} x: {tips[:, 0].mean():.3f} +- {tips[:, 0].std():.3f} "
        f"y: {tips[:, 1].mean():.3f} +- {tips[:, 1].std():.3f}"
    )

plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/fingertip_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Verify all 4 augmentations of each scene are in the same split.
NUM_SCENES = 32_560
violations = []

for scene_id in range(NUM_SCENES):
    scene_images = [scene_id + aug * NUM_SCENES for aug in range(4)]
    split_membership = set()

    for img_idx in scene_images:
        if img_idx in train_set:
            split_membership.add("train")
        elif img_idx in val_set:
            split_membership.add("val")
        elif img_idx in test_set:
            split_membership.add("test")

    if len(split_membership) > 1:
        violations.append((scene_id, split_membership))

if violations:
    print(f"[FAIL] {len(violations)} scenes have augmentations in different splits")
    for scene_id, membership in violations[:5]:
        print(f"  Scene {scene_id}: {sorted(membership)}")
else:
    print(f"[OK] All {NUM_SCENES:,} scenes have all 4 augmentations in the same split.")
    print("[OK] No data leakage detected.")